In [2]:
from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import pandas as pd
import traceback

# --------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------
INPUT_DIRS = {
    "ISIC": Path("../images_ISIC"),
    "MALIGNANT": Path("../imagenes_Malignant")
}

OUTPUT_DIR = Path("imagenes_RGB")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = "rgb_conversion_log.csv"

JPEG_QUALITY = 95
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

# --------------------------------------------------
# FUNCIÓN PRINCIPAL DE PROCESADO
# --------------------------------------------------
def process_image(img_path: Path, source: str):
    log_entry = {
        "image_name": img_path.name,
        "source": source,
        "status": "ok",
        "error": "",
        "overwritten": False
    }

    try:
        # Abrir imagen
        with Image.open(img_path) as img:
            # Corregir orientación EXIF
            img = ImageOps.exif_transpose(img)

            # Convertir a RGB (elimina alpha y grayscale)
            img = img.convert("RGB")

            # Convertir a numpy para asegurar tipo y rango
            img_np = np.array(img)

            if img_np.dtype != np.uint8:
                img_np = np.clip(img_np, 0, 255).astype(np.uint8)

            # Volver a PIL
            img = Image.fromarray(img_np, mode="RGB")

            output_path = OUTPUT_DIR / img_path.name

            if output_path.exists():
                log_entry["overwritten"] = True

            # Guardar como JPG
            img.save(output_path, format="JPEG", quality=JPEG_QUALITY)

    except Exception as e:
        log_entry["status"] = "failed"
        log_entry["error"] = str(e)
        print(f"[ERROR] {img_path.name}: {e}")
        traceback.print_exc()

    return log_entry

# --------------------------------------------------
# MAIN
# --------------------------------------------------
def main():
    logs = []

    for source, input_dir in INPUT_DIRS.items():
        if not input_dir.exists():
            print(f"[WARNING] Carpeta no encontrada: {input_dir}")
            continue

        image_files = [
            p for p in input_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
        ]

        print(f"[INFO] Procesando {len(image_files)} imágenes de {source}")

        for img_path in image_files:
            log_entry = process_image(img_path, source)
            logs.append(log_entry)

    # Guardar log
    df_log = pd.DataFrame(logs)
    df_log.to_csv(LOG_PATH, index=False)

    print("\n✅ Conversión a RGB finalizada")
    print(f"📄 Log guardado en: {LOG_PATH}")
    print(f"📂 Imágenes RGB en: {OUTPUT_DIR.resolve()}")

    # Resumen rápido
    print("\nResumen:")
    print(df_log["status"].value_counts())
    print("Sobrescritas:", df_log["overwritten"].sum())

if __name__ == "__main__":
    main()


[INFO] Procesando 401059 imágenes de ISIC
[INFO] Procesando 27055 imágenes de MALIGNANT

✅ Conversión a RGB finalizada
📄 Log guardado en: rgb_conversion_log.csv
📂 Imágenes RGB en: C:\TFG\src\imagenes_RGB

Resumen:
status
ok    428114
Name: count, dtype: int64
Sobrescritas: 393
